In [ ]:
%pip install -U \
  "autogluon.timeseries==1.5.0" \
  "torch>=2.6.0" \
  "torchvision>=0.21.0" \
  "torchaudio>=2.6.0" \
  "transformers>=4.48.0" \
  "tokenizers>=0.21.0" \
  "numpy==1.26.4"

  Using cached torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
Using cached torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl (899.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB

In [3]:
import torch
import pandas as pd
import numpy as np
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from sklearn.metrics import r2_score,mean_squared_error

In [4]:
df = pd.read_csv("../data/training/bitola_final_data.csv")
df.head()

,timestamp,sensorId,city,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,...,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2023-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,58.333333,28.333333,13.666667,943.0,12.00,8.3,63.645833,...,8.3,6.98,0.000000,1.000000,-0.5,0.866025,-0.433884,-0.900969,0,1
1,2023-12-01 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.250000,13.750000,5.000000,943.0,12.00,9.3,64.750000,...,9.3,7.50,0.258819,0.965926,-0.5,0.866025,-0.433884,-0.900969,0,1
2,2023-12-01 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,60.000000,10.000000,4.500000,942.5,12.25,8.3,65.312500,...,8.3,6.74,0.500000,0.866025,-0.5,0.866025,-0.433884,-0.900969,0,1
3,2023-12-01 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.250000,9.750000,4.500000,942.0,13.00,9.4,64.854167,...,9.4,7.84,0.707107,0.707107,-0.5,0.866025,-0.433884,-0.900969,0,1
4,2023-12-01 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.500000,5.000000,2.000000,942.0,13.00,9.0,65.062500,...,9.0,8.32,0.866025,0.500000,-0.5,0.866025,-0.433884,-0.900969,0,1


In [5]:
df.describe()

,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,...,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
count,192984.000000,149772.000000,149781.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,...,192984.000000,192984.000000,1.929840e+05,1.929840e+05,192984.000000,1.929840e+05,192984.000000,192984.000000,192984.000000,192984.000000
mean,52.712150,26.860537,14.470625,939.805608,16.794952,6.075046,52.241040,53.628155,51.232897,943.015291,...,6.173400,6.157256,-1.845999e-17,-5.550655e-17,-0.002785,-3.554140e-03,-0.002997,-0.000684,0.287278,0.414501
std,15.722700,49.673873,26.783313,12.797334,9.158462,3.743378,15.767958,16.392558,15.040098,6.651617,...,3.865857,3.848838,7.071086e-01,7.071086e-01,0.706861,7.073415e-01,0.707344,0.706866,0.452493,0.492637
min,9.500000,0.000000,0.000000,869.000000,-12.000000,0.000000,9.500000,9.500000,9.500000,911.000000,...,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-1.000000,-1.000000e+00,-0.974928,-0.900969,0.000000,0.000000
25%,40.759259,5.333333,2.333333,937.000000,9.694444,3.500000,40.466667,41.333333,39.750000,939.000000,...,3.500000,3.500000,-7.071068e-01,-7.071068e-01,-0.500000,-8.660254e-01,-0.781831,-0.900969,0.000000,0.000000
50%,53.604167,11.000000,5.750000,942.000000,16.000000,5.300000,53.000000,54.495833,52.250000,943.000000,...,5.400000,5.400000,6.123234e-17,-6.123234e-17,0.000000,-1.836970e-16,0.000000,-0.222521,0.000000,0.000000
75%,64.850000,27.000000,14.500000,946.000000,23.750000,7.740000,64.250000,66.000000,63.000000,947.000000,...,7.900000,7.900000,7.071068e-01,7.071068e-01,0.866025,5.000000e-01,0.781831,0.623490,1.000000,1.000000
max,99.000000,1995.000000,639.250000,971.750000,47.500000,32.800000,99.000000,99.000000,86.750000,971.750000,...,32.800000,32.800000,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,0.974928,1.000000,1.000000,1.000000


In [6]:
len(df)

192984

In [7]:
df['sensorId'].value_counts()

sensorId
16836a55-7140-43e2-9a63-56fac5cba714    17544
2002                                    17544
23b735ef-a996-4a7f-9998-2aa7e78827b0    17544
2819ecbb-5de3-4092-aa1d-4ba3a8c40add    17544
40f081a6-4095-43f7-bffb-64e2af8c026e    17544
7b316592-8036-41e2-b8dc-b06b6a9afd54    17544
874ff9c6-786d-45fc-a90e-48c7ffe03417    17544
87f82783-853b-417d-8964-b5cf11e44873    17544
d241a044-0a06-40c2-9d90-c91fd0a95060    17544
d7060523-b163-4cc3-bc94-970dac72a38a    17544
fec52a19-9148-4350-a1b4-ae0da05ee199    17544
Name: count, dtype: int64

In [8]:
df = df.drop(columns=['city'],axis=1)

In [9]:
TARGET = 'pm10'
ID_COL = 'sensorId'
TIME_COL = 'timestamp'
PREDICTION_LENGTH = 512

In [10]:
df[TIME_COL] = pd.to_datetime(df[TIME_COL])

In [11]:
df[TIME_COL] = df[TIME_COL].dt.tz_convert(None)

In [12]:
print(df["timestamp"].dtype)

datetime64[ns]


In [13]:
def clean_outliers_spatiotemporal(df, target_col='pm10', window_size=4, threshold=2.5):

    df = df.sort_values('timestamp')

    rolling_stats = df.set_index('timestamp')[target_col].rolling(window=f'{window_size}h')

    global_rolling_mean = rolling_stats.mean().reset_index(drop=True)
    global_rolling_std = rolling_stats.std().reset_index(drop=True)


    z_scores = (df[target_col] - global_rolling_mean) / (global_rolling_std + 1e-6)

    df[target_col] = df[target_col].mask(z_scores.abs() > threshold, global_rolling_mean)

    return df

In [14]:
df = clean_outliers_spatiotemporal(df)

In [15]:
df.describe()

,timestamp,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,...,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
count,192984,192984.000000,149772.000000,149781.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,...,192984.000000,192984.000000,1.929840e+05,1.929840e+05,192984.000000,1.929840e+05,192984.000000,192984.000000,192984.000000,192984.000000
mean,2024-11-30 11:30:00,52.712150,15.646266,14.470625,939.805608,16.794952,6.075046,52.241040,53.628155,51.232897,...,6.173400,6.157256,-1.892483e-17,-5.550425e-17,-0.002785,-3.554140e-03,-0.002997,-0.000684,0.287278,0.414501
min,2023-12-01 00:00:00,9.500000,0.000000,0.000000,869.000000,-12.000000,0.000000,9.500000,9.500000,9.500000,...,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-1.000000,-1.000000e+00,-0.974928,-0.900969,0.000000,0.000000
25%,2024-05-31 17:45:00,40.759259,5.250000,2.333333,937.000000,9.694444,3.500000,40.466667,41.333333,39.750000,...,3.500000,3.500000,-7.071068e-01,-7.071068e-01,-0.500000,-8.660254e-01,-0.781831,-0.900969,0.000000,0.000000
50%,2024-11-30 11:30:00,53.604167,9.584007,5.750000,942.000000,16.000000,5.300000,53.000000,54.495833,52.250000,...,5.400000,5.400000,6.123234e-17,-6.123234e-17,0.000000,-1.836970e-16,0.000000,-0.222521,0.000000,0.000000
75%,2025-06-01 05:15:00,64.850000,19.000000,14.500000,946.000000,23.750000,7.740000,64.250000,66.000000,63.000000,...,7.900000,7.900000,7.071068e-01,7.071068e-01,0.866025,5.000000e-01,0.781831,0.623490,1.000000,1.000000
max,2025-11-30 23:00:00,99.000000,507.500000,639.250000,971.750000,47.500000,32.800000,99.000000,99.000000,86.750000,...,32.800000,32.800000,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,0.974928,1.000000,1.000000,1.000000
std,NaN,15.722700,19.536842,26.783313,12.797334,9.158462,3.743378,15.767958,16.392558,15.040098,...,3.865857,3.848838,7.071086e-01,7.071086e-01,0.706861,7.073415e-01,0.707344,0.706866,0.452493,0.492637


In [16]:
data = TimeSeriesDataFrame.from_data_frame(
    df,
    id_column=ID_COL,
    timestamp_column=TIME_COL
)

In [17]:
data.columns

Index(['humidity', 'pm10', 'pm25', 'pressure', 'temperature', 'wind_speed',
       'neighbor1_humidity', 'neighbor2_humidity', 'neighbor3_humidity',
       'neighbor1_pressure', 'neighbor2_pressure', 'neighbor3_pressure',
       'neighbor1_temperature', 'neighbor2_temperature',
       'neighbor3_temperature', 'neighbor1_wind_speed', 'neighbor2_wind_speed',
       'neighbor3_wind_speed', 'hour_sin', 'hour_cos', 'month_sin',
       'month_cos', 'day_sin', 'day_cos', 'is_weekend', 'is_heating_season'],
      dtype='object')

In [18]:
train_data, test_data = data.train_test_split(prediction_length=PREDICTION_LENGTH)

Sorting the dataframe index before generating the train/test split.


In [19]:
predictor = TimeSeriesPredictor(
    target= TARGET,
    prediction_length=PREDICTION_LENGTH,
    eval_metric="MASE",
)

In [ ]:
predictor.fit(
    train_data,
    presets="chronos2",   # or "chronos2_small" for faster training
    time_limit=300
    verbosity=2
)

Beginning AutoGluon training... Time limit = 300s
AutoGluon will save models to '/mnt/c/Users/RazorVision/Desktop/project-vrnmp/offline-Phase/AutogluonModels/ag-20260516_180221'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          16
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 12.00/12.00 GB
Total GPU Memory:   Free: 12.00 GB, Allocated: 0.00 GB, Total: 12.00 GB
GPU Count:          1
Memory Avail:       10.35 GB / 15.58 GB (66.4%)
Disk Space Avail:   72.70 GB / 464.89 GB (15.6%)
Setting presets to: chronos2

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'hyperparameters': {'Chronos2': {'model_path': 'autogluon/chronos-2'}},
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 512,
 'quantile_levels': [0

KeyboardInterrupt: 

In [19]:
predictions = predictor.predict(train_data)

Model not specified in predict, will default to the model with the best validation score: Chronos2


In [20]:
print(predictions.head())

                                                               mean  \
item_id                              timestamp                        
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00  34.817322   
                                     2025-11-09 17:00:00  37.246620   
                                     2025-11-09 18:00:00  33.631908   
                                     2025-11-09 19:00:00  30.262213   
                                     2025-11-09 20:00:00  28.418341   

                                                                0.1  \
item_id                              timestamp                        
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00  24.643730   
                                     2025-11-09 17:00:00  24.688066   
                                     2025-11-09 18:00:00  21.267815   
                                     2025-11-09 19:00:00  18.245241   
                                     2025-11-09 20:00:00  16.805607   

    

In [21]:
performance = predictor.evaluate(test_data)

print(performance)

Model not specified in predict, will default to the model with the best validation score: Chronos2


{'MASE': -1.0510433854216212}


In [22]:
test_df = test_data.to_data_frame()
pred_df = predictions.to_data_frame()

In [23]:
merged = test_df.merge(
    pred_df[["mean"]],
    on=["item_id", "timestamp"],
    how="inner"
)

In [24]:
merged.columns

Index(['humidity', 'pm10', 'pm25', 'pressure', 'temperature', 'wind_speed',
       'neighbor1_humidity', 'neighbor2_humidity', 'neighbor3_humidity',
       'neighbor1_pressure', 'neighbor2_pressure', 'neighbor3_pressure',
       'neighbor1_temperature', 'neighbor2_temperature',
       'neighbor3_temperature', 'neighbor1_wind_speed', 'neighbor2_wind_speed',
       'neighbor3_wind_speed', 'hour_sin', 'hour_cos', 'month_sin',
       'month_cos', 'day_sin', 'day_cos', 'is_weekend', 'is_heating_season',
       'mean'],
      dtype='object')

In [25]:
merged = merged.rename(columns={"mean": "predicted"})

In [26]:
merged

humidity       pm10  \
item_id                              timestamp                                  
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00     71.25  44.500000   
                                     2025-11-09 17:00:00     72.00  32.333333   
                                     2025-11-09 18:00:00     73.00  26.750000   
                                     2025-11-09 19:00:00     73.75  10.500000   
                                     2025-11-09 20:00:00     73.25   9.500000   
...                                                            ...        ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-30 19:00:00     61.75  11.250000   
                                     2025-11-30 20:00:00     63.25   9.000000   
                                     2025-11-30 21:00:00     63.50  12.000000   
                                     2025-11-30 22:00:00     63.75   3.250000   
                                     2025-11-30 23:00:00     63.75   1.750000   

                                                               pm25  pressure  \
item_id                              timestamp                                  
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00  29.750000    944.00   
                                     2025-11-09 17:00:00  20.666667    944.00   
                                     2025-11-09 18:00:00  16.750000    944.25   
                                     2025-11-09 19:00:00   6.250000    945.00   
                                     2025-11-09 20:00:00   6.250000    945.00   
...                                                             ...       ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-30 19:00:00   6.000000    942.00   
                                     2025-11-30 20:00:00   5.666667    942.00   
                                     2025-11-30 21:00:00   6.000000    942.50   
                                     2025-11-30 22:00:00   2.500000    943.00   
                                     2025-11-30 23:00:00   0.750000    943.00   

                                                          temperature  \
item_id                              timestamp                          
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00        16.00   
                                     2025-11-09 17:00:00        16.00   
                                     2025-11-09 18:00:00        15.75   
                                     2025-11-09 19:00:00        15.00   
                                     2025-11-09 20:00:00        15.00   
...                                                               ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-30 19:00:00         9.00   
                                     2025-11-30 20:00:00         9.00   
                                     2025-11-30 21:00:00         8.25   
                                     2025-11-30 22:00:00         7.75   
                                     2025-11-30 23:00:00         7.00   

                                                          wind_speed  \
item_id                              timestamp                         
16836a55-7140-43e2-9a63-56fac5cba714 2025-11-09 16:00:00         5.1   
                                     2025-11-09 17:00:00         5.6   
                                     2025-11-09 18:00:00         1.5   
                                     2025-11-09 19:00:00         4.5   
                                     2025-11-09 20:00:00         4.0   
...                                                              ...   
fec52a19-9148-4350-a1b4-ae0da05ee199 2025-11-30 19:00:00         5.9   
                                     2025-11-30 20:00:00         5.7   
                                     2025-11-30 21:00:00         5.8   
                                     2025-11-30 22:00:00         5.6   
                                     2025-11-30 23:00:00         6.2   

                                                          neigh

In [27]:
merged.dropna(inplace=True)

In [28]:
y_true = merged["pm10"]
y_pred = merged["predicted"]

r2_global = r2_score(y_true, y_pred)
rmse_global = np.sqrt(mean_squared_error(y_true, y_pred))

print("Global R2:", r2_global)
print("Global RMSE:", rmse_global)

Global R2: 0.3080617372218424
Global RMSE: 14.770893507764018


In [29]:
per_sensor = merged.groupby("item_id").apply(
    lambda df: pd.Series({
        "r2": r2_score(df["pm10"], df["predicted"]),
        "rmse": np.sqrt(mean_squared_error(df["pm10"], df["predicted"]))
    })
)

print(per_sensor)

                                            r2       rmse
item_id                                                  
16836a55-7140-43e2-9a63-56fac5cba714  0.060030  19.696621
2002                                  0.040868  11.334386
23b735ef-a996-4a7f-9998-2aa7e78827b0 -0.002161   6.526225
2819ecbb-5de3-4092-aa1d-4ba3a8c40add  0.110113   6.960545
40f081a6-4095-43f7-bffb-64e2af8c026e  0.077893  13.782003
7b316592-8036-41e2-b8dc-b06b6a9afd54  0.196434  36.167436
874ff9c6-786d-45fc-a90e-48c7ffe03417  0.131498   9.232803
87f82783-853b-417d-8964-b5cf11e44873 -0.199955   9.288192
d241a044-0a06-40c2-9d90-c91fd0a95060  0.019788   3.705899
d7060523-b163-4cc3-bc94-970dac72a38a  0.041156   8.097631
fec52a19-9148-4350-a1b4-ae0da05ee199  0.113621   5.482459


In [30]:
predictor.path

'/Users/jovan/Documents/Kodovi/RNMP/Project/Offline-Phase/AutogluonModels/ag-20260509_182303'

In [31]:
import shutil

shutil.copytree(
    predictor.path,
    "chronos2_model_pm10_bitola"
)

'chronos2_model_pm10_bitola'